347. Top K Frequent Elements

Given an integer array nums and an integer k, return the k most frequent elements. You may return the answer in any order.

Example 1:

    Input: nums = [1,1,1,2,2,3], k = 2
    Output: [1,2]

Example 2:

    Input: nums = [1], k = 1
    Output: [1]

Example 3:

    Input: nums = [1,2,1,2,1,2,3,1,3,2], k = 2
    Output: [1,2]


Constraints:


Edge Cases:  

Clarifying Questions: 
    

Question might be framed as 
"Design a system to identify top-K heavy hitters from a data set"

OR 

"Imagine you’re working on a large-scale logging system at Meta. Each log entry contains an integer code representing a type of event. 

We want to continuously identify the K most frequent event codes in near real-time. Given an integer stream nums and a parameter k, design a system that returns the k most frequent elements. You may return the answer in any order."


"Sure. At its core, this is a frequency counting problem. The straightforward approach is:
- Use a hash map to count occurrences of each integer.
- Use a heap or bucket sort to extract the top K frequent elements.
The complexity depends on the extraction method: sorting all counts is O(n\log n), but using a min-heap of size K reduces it to O(n\log k). Bucket sort can achieve O(n) since frequencies are bounded by n."


In [ ]:
nums = [1,1,1,2,2,3] 
k = 2

seen = {}

for i in nums:
    if i not in seen:
        
        


In [2]:
# Approach 1. Hashmap + Sorting 

from collections import Counter 

def topKFrequent(nums, k):
    freq = Counter(nums) 
    # sort by frquency 
    sorted_items = sorted(freq.items(), key=lambda x: x[1], reverse = True)

    # extract top k  elements 
    return [num for num, _ in sorted_items[:k]]

nums = [1,1,1,2,2,2,3,3,3,4,5,6,7,6,8,9] 
k = 3
print (topKFrequent(nums, k))

[1, 2, 3]


In [3]:
# Approach #2- Min Heap (Optimal for k << 2) 
"""
Key insight
Keep heap size = k
Pop smallest frequency → retain top k
"""

from collections import Counter 
import heapq 

def topkFrequent(nums, k):
    freq = Counter(nums)
    heap = []
    for num, count in freq.item():
        heapq.heappush(heap, (count, num))

        if len(heap) > k:
            heapq.heappop(heap)

    return [num for _, num in heap]

nums = [1,1,1,2,2,3] 
k = 2
print (topKFrequent(nums, k))



[1, 2]


In [1]:
# Approach 3: Bucket Sort 
from collections import Counter

def topKFrequent(nums, k):
    freq = Counter(nums)
    
    # Create buckets: index = frequency
    buckets = [[] for _ in range(len(nums) + 1)]
    
    for num, count in freq.items():
        buckets[count].append(num)
    
    result = []
    
    # Traverse from highest frequency to lowest
    for i in range(len(buckets) - 1, 0, -1):
        for num in buckets[i]:
            result.append(num)
            if len(result) == k:
                return result

nums = [1,1,1,2,2,3] 
k = 2
print (topKFrequent(nums, k))

[1, 2]


In [ ]:
# Approach #4 Quick Select (Advanced)

from collections import Counter
import random

def topKFrequent(nums, k):
    freq = Counter(nums)
    unique = list(freq.keys())
    
    def partition(left, right, pivot_index):
        pivot_freq = freq[unique[pivot_index]]
        
        # Move pivot to end
        unique[pivot_index], unique[right] = unique[right], unique[pivot_index]
        
        store_index = left
        
        for i in range(left, right):
            if freq[unique[i]] < pivot_freq:
                unique[store_index], unique[i] = unique[i], unique[store_index]
                store_index += 1
        
        # Move pivot to final place
        unique[right], unique[store_index] = unique[store_index], unique[right]
        
        return store_index
    
    def quickselect(left, right, k_smallest):
        if left == right:
            return
        
        pivot_index = random.randint(left, right)
        pivot_index = partition(left, right, pivot_index)
        
        if k_smallest == pivot_index:
            return
        elif k_smallest < pivot_index:
            quickselect(left, pivot_index - 1, k_smallest)
        else:
            quickselect(pivot_index + 1, right, k_smallest)
    
    n = len(unique)
    
    # kth largest → (n - k)th smallest
    quickselect(0, n - 1, n - k)
    
    return unique[n - k:]



What the interviewer is probing
- Algorithmic mastery: Can you explain the optimal solution (hash map + heap / bucket sort) and its complexity?

- Scalability: How would your solution adapt if nums has billions of entries? What if it’s a stream instead of a static array?

- Trade-offs: Heap vs bucket sort vs quickselect. Memory vs speed. Approximate vs exact answers.

- System design thinking: How would you build this in a distributed environment? Would you use MapReduce, streaming frameworks (Flink, Spark), or probabilistic data structures (Count-Min Sketch)?

- Communication: Can you clearly articulate your reasoning, constraints, and assumptions?

Example E6-level discussion points
- Complexity analysis: Show why the naive sort is O(n\log n), but heap/bucket approaches can achieve O(n).

- Streaming adaptation: Discuss how you’d maintain top-K in a sliding window using min-heaps or sketches.

- Distributed systems: How to aggregate counts across shards? How to handle skew (e.g., one element dominating)?

- Fault tolerance: What happens if a node fails while aggregating counts?

- Approximation: When exact answers are too costly, how would you use algorithms like Misra-Gries or Count-Min Sketch?


In [ ]:
from collections import Counter
import heapq

def topKFrequent(nums, k):
    count = Counter(nums)
    return [item for item, _ in heapq.nlargest(k, count.items(), key=lambda x: x[1])]

Questions:

 ### "How would you handle this if the array had billions of elements?"

Great question. At that scale, we can’t fit everything in memory. I’d think in terms of distributed systems:
- Partition the data across shards using a hash function.
- Each shard computes local frequency counts.
- We then aggregate counts in a reduce step (MapReduce style).
- Finally, we run a distributed top-K algorithm, often using a min-heap per node and merging results.
This scales horizontally and handles billions of entries."

### "What if the data is streaming, not static?"

"For streaming, we need approximate algorithms. Exact counting is too expensive. I’d use:
- A Count-Min Sketch to approximate frequencies with sublinear memory.
- A Misra-Gries algorithm to maintain candidate top-K elements with bounded space.
- These give us near-real-time answers with controllable error bounds.
If exactness is required, we’d need a sliding window + min-heap approach, but that’s costly."

### What trade-offs would you highlight?"

"Heap-based solutions are memory efficient but slower for very large K. Bucket sort is faster but requires more memory. 

Approximate algorithms trade accuracy for scalability. In production, I’d choose based on latency requirements, memory constraints, and whether exactness is critical. 

For example, in a logging system, approximate answers are often fine."


### "How would you ensure fault tolerance in your distributed solution?"

Candidate:
"I’d design the aggregation step to be idempotent and checkpointed. If a node fails, its shard can be recomputed or replayed from logs. Using frameworks like Flink or Spark Streaming, we get built-in fault tolerance with state snapshots. This ensures resilience without manual intervention."

Interviewer:
"That’s a solid answer. Thank you."

👉 Notice how the candidate starts with the algorithm, then expands into scalability, streaming, distributed systems, and trade-offs. That’s exactly the depth expected at E6 — not just solving the problem, but showing how you’d build it into a real system at Meta scale.

Would you like me to also show how the candidate might sketch out pseudocode plus a high-level system diagram during the interview? That’s often part of demonstrating clarity at E6.



In [ ]:
def topKFrequent(nums, k):
    # Step 1: Count frequencies
    freq_map = {}
    for num in nums:
        freq_map[num] = freq_map.get(num, 0) + 1

    # Step 2: Use a min-heap of size k
    import heapq
    heap = []
    for num, freq in freq_map.items():
        heapq.heappush(heap, (freq, num))
        if len(heap) > k:
            heapq.heappop(heap)

    # Step 3: Extract results
    return [num for freq, num in heap]

- Time complexity: O(n\log k)
- Space complexity: O(n)
This is the local solution — what you’d run on a single machine.


System Diagram (Distributed / Streaming Context)

                ┌───────────────┐
                │   Event Logs   │
                └───────┬───────┘
                        │
                ┌───────▼────────┐
                │   Ingestion     │  (Kafka / PubSub)
                └───────┬────────┘
                        │
        ┌───────────────┼────────────────┐
        │               │                │
 ┌──────▼──────┐ ┌──────▼──────┐ ┌──────▼──────┐
 │ Shard Node 1│ │ Shard Node 2│ │ Shard Node N│
 │ (local freq │ │ (local freq │ │ (local freq │
 │   counting) │ │   counting) │ │   counting) │
 └──────┬──────┘ └──────┬──────┘ └──────┬──────┘
        │               │                │
        └───────────────┼────────────────┘
                        │
                ┌───────▼────────┐
                │ Aggregator     │  (Reduce step)
                │  - Merge heaps │
                │  - Compute topK│
                └───────┬────────┘
                        │
                ┌───────▼────────┐
                │   Dashboard    │
                │   or API       │
                └───────────────┘

"So the pseudocode shows the algorithmic foundation. At scale, I’d shard the data across nodes, each maintaining local frequency counts with a min-heap. Then an aggregator merges results to compute the global top-K. For streaming, I’d replace exact counts with Count-Min Sketch or Misra-Gries to reduce memory. The ingestion layer could be Kafka, and the aggregation could run on Flink or Spark Streaming. This design ensures scalability, fault tolerance, and near real-time results."

Monitoring & Alerting Layer
Once the system is producing top-K frequent elements, the candidate might say:
"In production, we don’t just want the top-K results — we want to know when something unusual happens. For example, if a rare error code suddenly spikes into the top-K, that could indicate a system failure or attack. So I’d add a monitoring and alerting layer."

Monitoring Metrics
- Frequency distribution trends: Track how counts change over time.
- Rate of change (derivative): Alert if an element’s frequency grows faster than a threshold.
- Unexpected entries in top-K: Flag if a new element appears in the top-K that wasn’t seen before.
- Skew detection: Alert if one element dominates (e.g., >90% of events).

Alerting Mechanisms
- Threshold-based alerts: e.g., “If error code 500 enters top-3, trigger an alert.”
- Anomaly detection models: Use statistical methods (z-score, moving averages) or ML-based anomaly detection.
- Integration with Ops tools: Send alerts to PagerDuty, Slack, or internal dashboards.

Extended System Diagram (with Monitoring)
                ┌───────────────┐
                │   Event Logs   │
                └───────┬───────┘
                        │
                ┌───────▼────────┐
                │   Ingestion     │
                └───────┬────────┘
                        │
        ┌───────────────┼────────────────┐
        │               │                │
 ┌──────▼──────┐ ┌──────▼──────┐ ┌──────▼──────┐
 │ Shard Node 1│ │ Shard Node 2│ │ Shard Node N│
 │ (local freq │ │ (local freq │ │ (local freq │
 │   counting) │ │   counting) │ │   counting) │
 └──────┬──────┘ └──────┬──────┘ └──────┬──────┘
        │               │                │
        └───────────────┼────────────────┘
                        │
                ┌───────▼────────┐
                │ Aggregator     │
                │  - Merge heaps │
                │  - Compute topK│
                └───────┬────────┘
                        │
        ┌───────────────┼────────────────┐
        │               │                │
 ┌──────▼──────┐ ┌──────▼────────┐ ┌──────▼────────┐
 │ Dashboard   │ │ Monitoring     │ │ Alert System  │
 │ (API/UI)    │ │ (metrics, ML)  │ │ (PagerDuty,   │
 │             │ │ anomaly detect)│ │ Slack, etc.)  │
 └─────────────┘ └────────────────┘ └────────────────┘



Candidate Commentary
"This monitoring layer ensures we don’t just compute top-K, but also detect anomalies in real time. For example, if a new error code suddenly spikes, the system can automatically trigger alerts to engineers. This closes the loop from raw data → insight → action, which is critical at Meta scale."

👉 At E6, this kind of forward-thinking operational design is what interviewers love — it shows you’re not just coding, but building systems that are robust, observable, and actionable.

Would you like me to simulate how the candidate might wrap up their answer — summarizing algorithm, system design, and monitoring in a concise “executive-level” conclusion? That’s often the final step in a strong interview performance.

